# Home vs Away Analysis

Este notebook tem como objetivo analisar e comparar o desempenho das equipes da NBA jogando em casa e fora de casa.

Nesta etapa, serão calculadas métricas que permitam identificar quais equipes apresentam melhor aproveitamento como mandantes e como visitantes.

In [20]:
import pandas as pd

In [21]:
games = pd.read_csv('../data/processed/clean_games.csv')
teams = pd.read_csv('../data/processed/clean_teams.csv')

## Visualização inicial dos dados

Nesta etapa, será feita uma revisão inicial dos datasets tratados que serão utilizados na análise principal.

In [22]:
games.head()

,GAME_DATE_EST,HOME_TEAM_ID,VISITOR_TEAM_ID,SEASON,PTS_home,PTS_away,HOME_TEAM_WINS
0,2022-12-22,1610612740,1610612759,2022,126.0,117.0,1
1,2022-12-22,1610612762,1610612764,2022,120.0,112.0,1
2,2022-12-21,1610612739,1610612749,2022,114.0,106.0,1
3,2022-12-21,1610612755,1610612765,2022,113.0,93.0,1
4,2022-12-21,1610612737,1610612741,2022,108.0,110.0,0


In [23]:
teams.head()

,TEAM_ID,ABBREVIATION,NICKNAME,CITY
0,1610612737,ATL,Hawks,Atlanta
1,1610612738,BOS,Celtics,Boston
2,1610612740,NOP,Pelicans,New Orleans
3,1610612741,CHI,Bulls,Chicago
4,1610612742,DAL,Mavericks,Dallas


## Criação da base de desempenho em casa

Aqui serão reunidas, para cada equipe mandante, informações como quantidade de jogos em casa, vitórias em casa e média de pontos marcados em casa.

In [24]:
home_performance = games.groupby('HOME_TEAM_ID').agg(
    home_games=('HOME_TEAM_ID', 'count'),
    home_wins=('HOME_TEAM_WINS', 'sum'),
    average_home_points=('PTS_home', 'mean')
).reset_index()

In [25]:
home_performance['home_win_percentage'] = home_performance['home_wins'] / home_performance['home_games']
home_performance.head().round(2)

,HOME_TEAM_ID,home_games,home_wins,average_home_points,home_win_percentage
0,1610612737,881,504,102.84,0.57
1,1610612738,946,603,102.65,0.64
2,1610612739,918,554,101.91,0.60
3,1610612740,848,464,102.28,0.55
4,1610612741,893,517,100.55,0.58


## Criação da base de desempenho fora de casa

Nesta etapa, será criada a base de desempenho das equipes como visitantes, incluindo quantidade de jogos fora, vitórias fora e média de pontos marcados fora de casa.

In [26]:
games['AWAY_TEAM_WINS'] = 1 - games['HOME_TEAM_WINS']

In [27]:
away_performance = games.groupby('VISITOR_TEAM_ID').agg(
    away_games=('VISITOR_TEAM_ID', 'count'),
    away_wins=('AWAY_TEAM_WINS', 'sum'),
    average_pts_away=('PTS_away', 'mean')
).reset_index()

In [28]:
away_performance['away_win_percentage'] = away_performance['away_wins'] / away_performance['away_games']
away_performance.head().round(2)

,VISITOR_TEAM_ID,away_games,away_wins,average_pts_away,away_win_percentage
0,1610612737,897,327,99.15,0.36
1,1610612738,932,452,100.91,0.48
2,1610612739,891,370,99.28,0.42
3,1610612740,870,326,99.71,0.37
4,1610612741,868,368,99.18,0.42


## Junção das bases de desempenho

Agora serão unidas as informações de desempenho em casa e fora de casa em uma única tabela por equipe.

In [29]:
performance = pd.merge(
    home_performance,
    away_performance,
    left_on='HOME_TEAM_ID',
    right_on='VISITOR_TEAM_ID',
    how='inner'
)

In [30]:
performance = performance.drop(columns=['VISITOR_TEAM_ID'])
performance = performance.rename(columns={'HOME_TEAM_ID': 'TEAM_ID'})
performance.head().round(2)

,TEAM_ID,home_games,home_wins,average_home_points,home_win_percentage,away_games,away_wins,average_pts_away,away_win_percentage
0,1610612737,881,504,102.84,0.57,897,327,99.15,0.36
1,1610612738,946,603,102.65,0.64,932,452,100.91,0.48
2,1610612739,918,554,101.91,0.60,891,370,99.28,0.42
3,1610612740,848,464,102.28,0.55,870,326,99.71,0.37
4,1610612741,893,517,100.55,0.58,868,368,99.18,0.42


## Associação com os nomes das equipes

Nesta etapa, a tabela de desempenho será associada ao dataset de equipes para exibir nomes e abreviações dos times.

In [31]:
performance = pd.merge(
    performance,
    teams[['TEAM_ID', 'NICKNAME', 'CITY']],
    on='TEAM_ID',
    how='left'
)

In [32]:
performance.head().round(2)

,TEAM_ID,home_games,home_wins,average_home_points,home_win_percentage,away_games,away_wins,average_pts_away,away_win_percentage,NICKNAME,CITY
0,1610612737,881,504,102.84,0.57,897,327,99.15,0.36,Hawks,Atlanta
1,1610612738,946,603,102.65,0.64,932,452,100.91,0.48,Celtics,Boston
2,1610612739,918,554,101.91,0.60,891,370,99.28,0.42,Cavaliers,Cleveland
3,1610612740,848,464,102.28,0.55,870,326,99.71,0.37,Pelicans,New Orleans
4,1610612741,893,517,100.55,0.58,868,368,99.18,0.42,Bulls,Chicago


## Cálculo da diferença de aproveitamento

Será criada uma métrica de diferença entre o aproveitamento em casa e o aproveitamento fora de casa, permitindo identificar quais equipes dependem mais do fator casa.

In [33]:
performance['win_percentage_difference'] = performance['home_win_percentage'] - performance['away_win_percentage']
performance.head().round(2)

,TEAM_ID,home_games,home_wins,average_home_points,home_win_percentage,away_games,away_wins,average_pts_away,away_win_percentage,NICKNAME,CITY,win_percentage_difference
0,1610612737,881,504,102.84,0.57,897,327,99.15,0.36,Hawks,Atlanta,0.21
1,1610612738,946,603,102.65,0.64,932,452,100.91,0.48,Celtics,Boston,0.15
2,1610612739,918,554,101.91,0.60,891,370,99.28,0.42,Cavaliers,Cleveland,0.19
3,1610612740,848,464,102.28,0.55,870,326,99.71,0.37,Pelicans,New Orleans,0.17
4,1610612741,893,517,100.55,0.58,868,368,99.18,0.42,Bulls,Chicago,0.15


## Ranking das equipes com melhor desempenho em casa

Nesta etapa, serão identificadas as equipes com maior taxa de vitória jogando em casa.

In [34]:
performance[['CITY', 'NICKNAME', 'home_games', 'home_wins', 'home_win_percentage']].sort_values(
    by='home_win_percentage',
    ascending=False
).head(10).round(2)

,CITY,NICKNAME,home_games,home_wins,home_win_percentage
22,San Antonio,Spurs,936,684,0.73
6,Denver,Nuggets,869,588,0.68
7,Golden State,Warriors,904,606,0.67
11,Miami,Heat,955,632,0.66
25,Utah,Jazz,868,573,0.66
5,Dallas,Mavericks,904,594,0.66
1,Boston,Celtics,946,603,0.64
8,Houston,Rockets,895,570,0.64
17,Indiana,Pacers,886,550,0.62
20,Portland,Trail Blazers,867,524,0.60


## Ranking das equipes com melhor desempenho fora de casa

Agora serão identificadas as equipes com maior taxa de vitória como visitantes.

In [35]:
performance[['CITY', 'NICKNAME', 'away_games', 'away_wins', 'away_win_percentage']].sort_values(
    by='away_win_percentage',
    ascending=False
).head(10).round(2)

,CITY,NICKNAME,away_games,away_wins,away_win_percentage
22,San Antonio,Spurs,927,498,0.54
5,Dallas,Mavericks,912,443,0.49
1,Boston,Celtics,932,452,0.48
8,Houston,Rockets,893,425,0.48
11,Miami,Heat,936,441,0.47
19,Phoenix,Suns,877,401,0.46
7,Golden State,Warriors,910,413,0.45
6,Denver,Nuggets,913,405,0.44
23,Oklahoma City,Thunder,890,391,0.44
24,Toronto,Raptors,878,382,0.44


## Equipes com maior diferença entre casa e fora

A seguir, serão observadas as equipes que apresentam maior diferença entre o aproveitamento em casa e o aproveitamento fora, indicando maior impacto do fator casa.

In [36]:
performance[['CITY', 'NICKNAME', 'home_win_percentage', 'away_win_percentage', 'win_percentage_difference']].sort_values(
    by='win_percentage_difference',
    ascending=False
).head(10).round(2)

,CITY,NICKNAME,home_win_percentage,away_win_percentage,win_percentage_difference
25,Utah,Jazz,0.66,0.42,0.24
6,Denver,Nuggets,0.68,0.44,0.23
20,Portland,Trail Blazers,0.60,0.39,0.22
7,Golden State,Warriors,0.67,0.45,0.22
17,Indiana,Pacers,0.62,0.41,0.21
0,Atlanta,Hawks,0.57,0.36,0.21
12,Milwaukee,Bucks,0.58,0.38,0.20
27,Washington,Wizards,0.53,0.34,0.19
26,Memphis,Grizzlies,0.60,0.41,0.19
22,San Antonio,Spurs,0.73,0.54,0.19


## Tabela final de desempenho

A tabela final reúne as principais métricas calculadas para cada equipe e servirá de base para a etapa de visualizações.

In [37]:
performance = performance[['TEAM_ID', 'CITY', 'NICKNAME',
                           'home_games', 'home_wins', 'home_win_percentage',
                           'away_games', 'away_wins', 'away_win_percentage',
                           'average_home_points', 'average_pts_away', 'win_percentage_difference']]

performance.head().round(2)

,TEAM_ID,CITY,NICKNAME,home_games,home_wins,home_win_percentage,away_games,away_wins,away_win_percentage,average_home_points,average_pts_away,win_percentage_difference
0,1610612737,Atlanta,Hawks,881,504,0.57,897,327,0.36,102.84,99.15,0.21
1,1610612738,Boston,Celtics,946,603,0.64,932,452,0.48,102.65,100.91,0.15
2,1610612739,Cleveland,Cavaliers,918,554,0.60,891,370,0.42,101.91,99.28,0.19
3,1610612740,New Orleans,Pelicans,848,464,0.55,870,326,0.37,102.28,99.71,0.17
4,1610612741,Chicago,Bulls,893,517,0.58,868,368,0.42,100.55,99.18,0.15


## Conclusões da análise casa vs fora

A análise permitiu comparar o desempenho das equipes em partidas como mandantes e visitantes, utilizando métricas como quantidade de jogos, vitórias, taxa de aproveitamento e média de pontos.

A partir dessas informações, é possível identificar quais equipes apresentam melhor desempenho em casa, quais mantêm bom desempenho fora de casa e quais parecem ser mais influenciadas pelo fator casa.

Esses resultados servirão como base para a etapa final do projeto, em que serão construídas visualizações para facilitar a interpretação dos dados.

In [38]:
performance.to_csv('../data/processed/team_home_away_performance.csv', index=False)